In [1]:
!pip -q install chromadb sentence-transformers pypdf ollama

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [4]:
client = chromadb.Client()

# Delete the collection if it already exists (avoids duplicate errors)
try:
    client.delete_collection("documents")
except:
    pass

collection = client.create_collection(name="documents")

print("Collection created successfully!")

Collection created successfully!


In [5]:
documents = [
    "Python is a programming language.",
    "Java is an object-oriented programming language.",
    "Machine Learning predicts patterns from data.",
    "Artificial Intelligence simulates human intelligence.",
    "Deep Learning uses neural networks.",
    "Natural Language Processing understands human language.",
    "Cloud Computing provides scalable resources.",
    "Docker is used for containerization.",
    "Git is a version control system.",
    "GitHub hosts software repositories.",
    "Ollama runs Large Language Models locally.",
    "ChromaDB is a vector database.",
    "FAISS is used for similarity search.",
    "Embeddings convert text into numerical vectors.",
    "Cosine similarity measures semantic similarity.",
    "LangChain helps build LLM applications.",
    "CrewAI creates AI agents.",
    "RAG combines retrieval with generation.",
    "Transformers understand contextual relationships.",
    "Data Science extracts insights from data."
]

ids = [f"id_{i}" for i in range(20)]

metadatas = [
    {"topic":"python"},
    {"topic":"java"},
    {"topic":"ml"},
    {"topic":"ai"},
    {"topic":"deep_learning"},
    {"topic":"nlp"},
    {"topic":"cloud"},
    {"topic":"docker"},
    {"topic":"git"},
    {"topic":"github"},
    {"topic":"ollama"},
    {"topic":"chromadb"},
    {"topic":"faiss"},
    {"topic":"embeddings"},
    {"topic":"similarity"},
    {"topic":"langchain"},
    {"topic":"crewai"},
    {"topic":"rag"},
    {"topic":"transformers"},
    {"topic":"datascience"}
]

print("20 documents created successfully!")

20 documents created successfully!


In [6]:
embeddings = embedding_model.encode(documents).tolist()

print(f"Generated {len(embeddings)} embeddings successfully!")

Generated 20 embeddings successfully!


In [7]:
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids,
    metadatas=metadatas
)

print("20 documents added to ChromaDB successfully!")

20 documents added to ChromaDB successfully!


In [8]:
query = "What is semantic search?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Top 3 Similar Documents:\n")

for i, doc in enumerate(results["documents"][0], start=1):
    print(f"{i}. {doc}")

Top 3 Similar Documents:

1. FAISS is used for similarity search.
2. Cosine similarity measures semantic similarity.
3. Natural Language Processing understands human language.


In [9]:
results = collection.get(
    where={"topic":"chromadb"}
)

print(results["documents"])

['ChromaDB is a vector database.']


## Manual Verification

Query:
What is semantic search?

Retrieved Documents:

1. Cosine similarity measures semantic similarity.
2. Embeddings convert text into numerical vectors.
3. ChromaDB is a vector database.

These retrieved documents are relevant to the query because semantic search relies on vector embeddings and cosine similarity to retrieve contextually similar documents.

In [10]:
from pypdf import PdfReader

reader = PdfReader("SANJANA C A_RESUME.pdf")

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print("PDF loaded successfully!")
print("\nFirst 500 characters:\n")
print(text[:500])

PDF loaded successfully!

First 500 characters:

SANJANA C A 
✉ sanjanaca622@gmail.com   |   📞 +91 9148132577   |   🔗 linkedin.com/in/sanjana-c-a-gcu3012-6a11762ab   |   
💻 github.com/Sanjanaca150 
PROFILE 
Aspiring Software Engineer and Information Science student with strong expertise in Java, Python, MySQL, and Data 
Structures. Skilled in developing CRUD applications, Python automation tools, and full -stack web projects. Seeking 
an engineering internship to apply technical problem -solving skills and gain hands -on software development 



In [11]:
chunk_size = 300

chunks = [
    text[i:i + chunk_size]
    for i in range(0, len(text), chunk_size)
]

print("Number of chunks:", len(chunks))

print("\nFirst Chunk:\n")
print(chunks[0])

Number of chunks: 7

First Chunk:

SANJANA C A 
✉ sanjanaca622@gmail.com   |   📞 +91 9148132577   |   🔗 linkedin.com/in/sanjana-c-a-gcu3012-6a11762ab   |   
💻 github.com/Sanjanaca150 
PROFILE 
Aspiring Software Engineer and Information Science student with strong expertise in Java, Python, MySQL, and Data 
Structures. Skilled in deve


In [12]:
try:
    client.delete_collection("resume_collection")
except:
    pass

pdf_collection = client.create_collection(
    name="resume_collection"
)

print("Resume collection created successfully!")

Resume collection created successfully!


In [13]:
chunk_embeddings = embedding_model.encode(chunks).tolist()

pdf_collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings
)

print("Chunks stored successfully in ChromaDB!")

Chunks stored successfully in ChromaDB!


In [14]:
question = "What technical skills does Sanjana have?"

query_embedding = embedding_model.encode(question).tolist()

results = pdf_collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

top_chunks = results["documents"][0]

print("Top 3 Retrieved Chunks:\n")

for i, chunk in enumerate(top_chunks, start=1):
    print(f"\nChunk {i}\n")
    print(chunk)

Top 3 Retrieved Chunks:


Chunk 1

SANJANA C A 
✉ sanjanaca622@gmail.com   |   📞 +91 9148132577   |   🔗 linkedin.com/in/sanjana-c-a-gcu3012-6a11762ab   |   
💻 github.com/Sanjanaca150 
PROFILE 
Aspiring Software Engineer and Information Science student with strong expertise in Java, Python, MySQL, and Data 
Structures. Skilled in deve

Chunk 2

loping CRUD applications, Python automation tools, and full -stack web projects. Seeking 
an engineering internship to apply technical problem -solving skills and gain hands -on software development 
experience. 
TECHNICAL SKILLS 
• Programming Languages: Java, SQL, Python 
• Database Technologies: 

Chunk 3

MySQL, DBMS 
• Web Technologies: HTML, CSS, JavaScript 
• Tools & Platforms: Git, GitHub, Eclipse IDE 
• Core Concepts: OOP, Data Structures, Exception Handling 
EDUCATION 
Garden City University, Bengaluru 
• Bachelor of Engineering (Information Science & Engineering) 
• Expected Graduation: 2027  


## Ollama Integration

The retrieved top-3 chunks were passed to the local Ollama model (`llama3.2:3b`) using the separate script `ollama_rag.py`.

The model successfully answered the question based on the retrieved context.

Refer to the attached screenshot of the Ollama output for verification.
